# Global RF
- Preprocess data
- Compare linear and non linear model
- Explore Clustering of residuals
- Spatial Crosvalidation

In [ ]:
import pathlib

import esda
import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyinterpolate
import shapely
import wkls
from libpysal import graph
from matplotlib.colors import ListedColormap
from matplotlib_map_utils.core.north_arrow import NorthArrow, north_arrow
from matplotlib_map_utils.core.scale_bar import ScaleBar, scale_bar
from shapely import wkb
from sklearn import ensemble, metrics, model_selection
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import StandardScaler

## Preprocess the data
Remove unnecessary column, devide the data by total population and standardize it.

In [ ]:
def process_file(path, path_total):
    cols_no_division = [
        "Hustota obyvatel na obytnou plochu",
        "Počet obyvatel na dům",
        "Počet obyvatel na byt",
    ]

    # Load total population data
    total = pd.read_csv(path_total, dtype={"nadzsjd": str}, index_col=0)

    # Load main data
    data = gpd.read_parquet(path)

    # Merge data
    data_total = data.join(total)

    # Drop unnecessary columns
    data_relative = data_total.drop(
        columns=[
            "NUTS_2",
            "naz_oblast",
            "NUTS_3",
            "naz_kraj",
            #    "kod_okres",
            "naz_okres",
            "naz_orp",
            "kod_orp",
            "kod_obec",
            "naz_obec",
            "kod_mco",
            "nazev_mco",
        ]
    )

    # Convert columns (except 'geometry') to float
    cols_numeric = data_relative.columns.drop(["geometry", "kod_okres"])
    data_relative[cols_numeric] = data_relative[cols_numeric].astype(float)

    # Normalize by total population (except certain columns)
    population = data_relative["Obyvatelstvo celkem"].replace(
        0, np.nan
    )  # avoid division by zero
    cols_to_normalize = [
        col
        for col in cols_numeric
        if col not in cols_no_division + ["Obyvatelstvo celkem"]
    ]
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].div(
        population, axis=0
    )

    # Drop rows with NaNs after division
    data_relative = data_relative.dropna(subset=cols_to_normalize)

    # Clip extremely large or small values
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].clip(-1e6, 1e6)
    data_relative[cols_no_division] = data_relative[cols_no_division].clip(-1e6, 1e6)

    # Replace any remaining infinities with NaN and drop them
    data_relative.replace([np.inf, -np.inf], np.nan, inplace=True)
    data_relative.dropna(subset=cols_to_normalize + cols_no_division, inplace=True)

    # Scale columns
    scaler = StandardScaler()
    data_relative[cols_to_normalize] = scaler.fit_transform(
        data_relative[cols_to_normalize]
    )
    data_relative[cols_no_division] = scaler.fit_transform(
        data_relative[cols_no_division]
    )

    return data_relative

In [ ]:
path = "/data/uscuni-restricted/04_spatial_census/_merged_census_2021.parquet"
path_total = "/data/uscuni-restricted/04_spatial_census/total.csv"

In [ ]:
data_relative = process_file(path, path_total)

In [ ]:
data_r = data_relative[data_relative.columns.drop(["Obyvatelstvo celkem", "kod_okres"])]
data_r.to_parquet(
    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
)

## Assign cluster label


In [ ]:
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = data_relative.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(
    [
        "geometry",
        "kod_nadzsj_d",
        "final_without_noise",
        "kod_okres",
        "Obyvatelstvo celkem",
    ]
)

data["Cluster"] = data["final_without_noise"].map(cluster_mapping[3])

In [ ]:
data["Cluster"].unique()

In [ ]:
fa = [
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání:  vysokoškolské - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý",
    "Počet osob v bytech celkem  s právním důvodem užívání: družstevní",
    "Počet obyvatel na byt",
    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba",
    "Obyvatelstvo - věk: 0 - 6  - celkem",
    "Obyvatelstvo - věk: 7 - 14  - celkem",
    "Obyvatelstvo - věk: 15 - 24  - celkem",
    "Obyvatelstvo - věk: 45 - 54  - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem",
    "Obyvatelstvo - s trvalým pobytem - celkem",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem",
    "Obyvatelstvo - rodinný stav: rozvedení - celkem",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem",
]

## Classification


In [ ]:
# Assign independent variables and the target
independent = data[fa]
target = data["Cluster"]
# target = (data["Cluster"] == 5).astype(int)

In [ ]:
cluster_colors = {
    1: "#4069BC",
    2: "#7CBAE4",
    3: "#E69C63",
    4: "#eec1d5",
    5: "#E0665F",
    6: "#ECBF43",
    7: "#b2cd32",
    8: "#1F943E",
}
data["cluster_color"] = data["Cluster"].map(cluster_colors)

In [ ]:
# Plot original data
ax = data.plot(
    color=data["cluster_color"],
    legend=True,
    figsize=(9, 9),
    markersize=0.1,
    categorical=True,
)
ax.set_axis_off()

In [ ]:
# Split data
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    independent,
    target,
    test_size=0.2,
    random_state=42,
)

### Linear model


In [ ]:
linear_model = LogisticRegression(
    n_jobs=-1, max_iter=5000, solver="saga", random_state=42, class_weight="balanced"
)
linear_model.fit(X_train, y_train)
linear_model.score(X_train, y_train), linear_model.score(X_test, y_test)

In [ ]:
y_pred_train = linear_model.predict(X_train)
y_pred_test = linear_model.predict(X_test)

f1_train = f1_score(y_train, y_pred_train, average="macro")
f1_test = f1_score(y_test, y_pred_test, average="macro")

f1_train, f1_test

### Non-Linear model


In [ ]:
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced")
rf_model.fit(X_train, y_train)
rf_model.score(X_train, y_train), rf_model.score(X_test, y_test)

#### Parameter tuning


In [ ]:
np.mean([estimator.tree_.max_depth for estimator in rf_model.estimators_])

In [ ]:
rf_model = ensemble.RandomForestClassifier(
    min_samples_split=10,
    min_samples_leaf=5,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
    n_estimators=300,
    max_features="log2",
    class_weight="balanced",
)
rf_model.fit(X_train, y_train)

In [ ]:
rf_model.score(X_train, y_train), rf_model.score(X_test, y_test)

In [ ]:
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

f1_train = f1_score(y_train, y_pred_train, average="macro")
f1_test = f1_score(y_test, y_pred_test, average="macro")

f1_train, f1_test

In [ ]:
pred = rf_model.predict(X_test)

In [ ]:
accuracy = metrics.accuracy_score(pred, y_test)
kappa = metrics.cohen_kappa_score(pred, y_test)

summary = f"""\
Evaluation metrics
==================
Basic model:
  Accuracy: {round(accuracy, 3)}
  Kappa:    {round(kappa, 3)}
"""

print(summary)

In [ ]:
predicted = model_selection.cross_val_predict(
    rf_model, independent, target, cv=4, n_jobs=-1
)

ax = data.plot(predicted, legend=True, figsize=(9, 9), markersize=0.1, categorical=True)
ax.set_axis_off()

In [ ]:
contiguity = graph.Graph.build_knn(data.centroid, k=30)
contiguity_b = contiguity.transform("b")

In [ ]:
for variable in independent:
    gs_global = esda.Moran(data[variable], contiguity_b)
    print(
        f"Moran's I of {variable} is {gs_global.I:.2f} with the  p-value of {gs_global.p_sim}."
    )

In [ ]:
for variable in independent:
    gs_global = esda.G(data[variable], contiguity_b)
    print(
        f"Global G of {variable} is {gs_global.G:.10f} with the  p-value of {gs_global.p_sim}."
    )

### Spatial Crossvalidation

In [ ]:
gkf = model_selection.StratifiedGroupKFold(n_splits=5)
splits = gkf.split(
    independent,
    target,
    groups=data.kod_okres,
)

In [ ]:
split_label = np.empty(len(data), dtype=float)
split_label

In [ ]:
for i, (_train_idx, test_idx) in enumerate(splits):
    split_label[test_idx] = i
data["split"] = split_label

In [ ]:
ax = data.plot("split", categorical=True, figsize=(9, 9), markersize=0.1, legend=True)
data.dissolve("kod_okres").convex_hull.boundary.plot(
    ax=ax, color="k", linewidth=0.5, markersize=0
)
ax.set_axis_off()

In [ ]:
train = data["split"] != 0
X_train = independent.loc[train]
y_train = data["Cluster"].loc[train]

test = data["split"] == 0
X_test = independent.loc[test]
y_test = data["Cluster"].loc[test]

In [ ]:
rf_spatial_cv = rf_model
rf_spatial_cv.fit(X_train, y_train)

In [ ]:
rf_spatial_cv.score(X_train, y_train), rf_spatial_cv.score(X_test, y_test)

In [ ]:
pred = rf_spatial_cv.predict(X_test)

accuracy_spatial_cv = metrics.accuracy_score(pred, y_test)
kappa_spatial_cv = metrics.cohen_kappa_score(pred, y_test)

summary += f"""\
Basic model with spatial cross-validation:
  Accuracy: {round(accuracy_spatial_cv, 3)}
  Kappa:    {round(kappa_spatial_cv, 3)}
"""

print(summary)

In [ ]:
report_dict = classification_report(y_test, pred, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

report_df

In [ ]:
predicted_spatial = model_selection.cross_val_predict(
    rf_spatial_cv, independent, target, cv=5, n_jobs=-1
)

In [ ]:
report_dict = classification_report(target, predicted_spatial, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

report_df = report_df.rename(
    index={
        "1": "Incoherent Large-Scale Homogeneous Fabric",
        "2": "Incoherent Large-Scale Heterogeneous Fabric",
        "3": "Incoherent Small-Scale Linear Fabric",
        "4": "Incoherent Small-Scale Sparse Fabric",
        "5": "Incoherent Small-Scale Compact Fabric",
        "6": "Coherent Interconnected Fabric",
        "7": "Coherent Dense Disjoint Fabric",
        "8": "Coherent Dense Adjacent Fabric",
    }
)
report_df.round(2)

In [ ]:
predicted_spatial = model_selection.cross_val_predict(
    linear_model, independent, target, cv=5, n_jobs=-1
)

In [ ]:
report_dict = classification_report(target, predicted_spatial, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df = report_df.rename(
    index={
        "1": "Incoherent Large-Scale Homogeneous Fabric",
        "2": "Incoherent Large-Scale Heterogeneous Fabric",
        "3": "Incoherent Small-Scale Linear Fabric",
        "4": "Incoherent Small-Scale Sparse Fabric",
        "5": "Incoherent Small-Scale Compact Fabric",
        "6": "Coherent Interconnected Fabric",
        "7": "Coherent Dense Disjoint Fabric",
        "8": "Coherent Dense Adjacent Fabric",
    }
)
report_df.round(2)

### Residual exploration

In [ ]:
my_colors = ["#E6323B", "#95D2ED"]
my_cmap = ListedColormap(my_colors)

In [ ]:
ax = data.plot(
    predicted_spatial == target,
    categorical=True,
    figsize=(15, 10),
    markersize=0.1,
    cmap=my_cmap,
    legend=True,
)
ax.set_axis_off()

In [ ]:
data["error"] = predicted_spatial == target
data["error"] = data["error"].astype(int).astype(float).values

In [ ]:
contiguity = graph.Graph.build_knn(data.centroid, k=90)
contiguity_b = contiguity.transform("b")

### Join Counts

In [ ]:
jc = esda.Join_Counts(data["error"], contiguity_b)


# Results
print("BB count:", jc.bb)  # 1–1 joins
print("WW count:", jc.ww)  # 0–0 joins
print("BW count:", jc.bw)
print("mean BB:", jc.mean_bb)
print("p-value:", jc.p_sim_bb)

#### Getirs Ord

In [ ]:
g_global = esda.G(data["error"], contiguity_b, permutations=999)
print(g_global.G)  # global G statistic
print(g_global.p_sim)

In [ ]:
g_local = esda.G_Local(data["error"], contiguity_b, star=True, permutations=999)
data["G_sig"] = g_local.p_sim < 0.05

In [ ]:
data["spot_type"] = "Insignificant"

data.loc[(g_local.p_sim < 0.05) & (g_local.Zs > 0), "spot_type"] = "Hot Spot"
data.loc[(g_local.p_sim < 0.05) & (g_local.Zs < 0), "spot_type"] = "Cold Spot"

data["spot_type"].value_counts()

In [ ]:
NorthArrow.set_size("small")
ScaleBar.set_size("small")

In [ ]:
f, ax = plt.subplots(figsize=(8, 10))
data.loc[data["spot_type"] == "Insignificant"].plot(ax=ax, color="lightgrey")
data.loc[data["spot_type"] == "Hot Spot"].plot(ax=ax, color="lightgrey")
data.loc[data["spot_type"] == "Cold Spot"].plot(ax=ax, color="#d7191c")

north_arrow(
    ax, location="upper left", rotation={"crs": data.crs, "reference": "center"}
)
scale_bar(
    ax, location="lower left", style="boxes", bar={"projection": data.crs, "unit": "km"}
)

ax.set_axis_off()

In [ ]:
contiguity = graph.Graph.build_fuzzy_contiguity(
    data.loc[data["spot_type"] == "Cold Spot"].geometry, buffer=10
)

In [ ]:
cl = contiguity.component_labels
cl.value_counts()

In [ ]:
_81 = data.loc[data["spot_type"] == "Cold Spot"][cl == 86]
_81.plot()

In [ ]:
morava = _81.buffer(1).to_frame("geometry").dissolve()

In [ ]:
morava

In [ ]:
morava = morava.explode()

In [ ]:
morava

In [ ]:
morava.exterior.plot()

In [ ]:
_0 = data.loc[data["spot_type"] == "Cold Spot"][cl == 0]
_0.plot()

In [ ]:
suburbs = _0.buffer(1).to_frame("geometry").dissolve()

In [ ]:
suburbs

In [ ]:
suburbs.exterior.plot()

In [ ]:
cz_wkb = wkls.cz.wkb()


geom = wkb.loads(cz_wkb)

In [ ]:
stat = gpd.GeoDataFrame({"name": ["CZ"]}, geometry=[geom], crs="EPSG:4326").to_crs(5514)

# plot
stat.exterior.plot()

In [ ]:
gpd.read_geojson("notebooks/04_results/stat.geojson")

In [ ]:
from matplotlib_scalebar.scalebar import ScaleBar

In [ ]:
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="#d7191c", edgecolor="none", label="Hot spot"),
    Patch(facecolor="lightgrey", edgecolor="none", label="Not significant"),
]

### Feature importance

In [ ]:
f, ax = plt.subplots(figsize=(8, 10))
data.loc[data["spot_type"] == "Insignificant"].plot(ax=ax, color="lightgrey")
data.loc[data["spot_type"] == "Hot Spot"].plot(ax=ax, color="lightgrey")
data.loc[data["spot_type"] == "Cold Spot"].plot(ax=ax, color="#d7191c")
# suburbs.exterior.plot(ax=ax, color="k", linewidth=0.8)
# morava.exterior.plot(ax=ax, color="k", linewidth=0.8)
stat.exterior.plot(ax=ax, color="k", linewidth=0.5)


ax.add_artist(ScaleBar(1, location="lower right", height_fraction=0.005))


# ax.annotate("Moravian Region",
#             xy=(-555000, -1151000),
#             # xycoords="data",
#             # xytext=(-530000, -1021000),
#             textcoords='data',
#             arrowprops=dict(arrowstyle="-", connectionstyle="arc3, rad=-0.1"))

# # ax.annotate("Prague Suburbs",
# #             xy=(-725000, -1051000),
# #             xycoords="data",
#             xytext=(-650000, -951000),
#             textcoords='data',
#             arrowprops=dict(arrowstyle="-", connectionstyle="arc3, rad=-0.2"))

ax.legend()


ax.legend(
    handles=legend_elements,
    loc="lower left",
    # bbox_to_anchor=(0, -0.03),
    frameon=False,
)

ax.set_axis_off()
# Local spatial significance of incorrect predictions
f.savefig("cz_correlation.png", dpi=300, bbox_inches="tight")

In [ ]:
feat_importances = pd.Series(
    rf_model.feature_importances_, index=X_train.columns
).sort_values()
plt.figure(figsize=(5, 20))
plt.axvline(x=0.01, color="red", linestyle="--", linewidth=1)


feat_importances.plot(kind="barh")